# Figure 10: MLP Shapley Analysis Feature Importance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

import torch
import shap

plt.rcParams['font.family'] = 'Times New Roman'

# Load Models and Data

In [ ]:
nn_fval_path = './data/model_FValue.pt'
nn_w1_path = './data/model_W1.pt'

nn_fval = torch.load(nn_fval_path)
nn_w1 = torch.load(nn_w1_path)

nn_fval.eval()
nn_w1.eval()

test_filename = './data/data_bias_e5_1.csv'
df = pd.read_csv(test_filename)

input_feats = ['gamma1', 'lambda1', 'delta', 'epsilon', 'NRow', 'NCol']

X = df[input_feats].values

PredictorScaler = StandardScaler()
PredScaleFit = PredictorScaler.fit(X)
X_scaled = PredScaleFit.transform(X)

# Compute SHAP Values

In [ ]:
class NN_wrapper:
    def __init__(self, model):
        self.model = model

    def predict(self, X):
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_preds_tensor = self.model(X_tensor)
        y_preds = y_preds_tensor.detach().numpy()
        return y_preds

nn_fval_wrapper = NN_wrapper(nn_fval)
nn_w1_wrapper = NN_wrapper(nn_w1)

explainer_fval = shap.Explainer(nn_fval_wrapper.predict, shap.sample(X_scaled, 100))
shap_values_fval = explainer_fval(X_scaled)

explainer_w1 = shap.Explainer(nn_w1_wrapper.predict, shap.sample(X_scaled, 100))
shap_values_w1 = explainer_w1(X_scaled)

# Beeswarm Plots (Figure 10a)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

plt.sca(axs[0])
shap.plots.beeswarm(shap_values_fval, show=False)
axs[0].set_title(r'$\phi_{\tau}$', fontsize=16)

plt.sca(axs[1])
shap.plots.beeswarm(shap_values_w1, show=False)
axs[1].set_title(r'$\Delta_{\tau}$', fontsize=16)

fig.suptitle('MLP Shapley analysis feature importance', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('Figure10.png', dpi=300, bbox_inches='tight')
plt.show()

# Mean Absolute SHAP Values (Figure 10b)

In [ ]:
feature_labels = [r'$\gamma$', r'$\lambda$', r'$\delta$', r'$\epsilon$', r'$N_1$', r'$N_2$']

print('phi_tau mean |SHAP|:')
for i in range(6):
    print(f'  {feature_labels[i]}: {np.mean(np.abs(shap_values_fval.values[:,i])):.3f}')

print('\nDelta_tau mean |SHAP|:')
for i in range(6):
    print(f'  {feature_labels[i]}: {np.mean(np.abs(shap_values_w1.values[:,i])):.3f}')